In [ ]:
import os
import cv2
import numpy as np
import findspark
from pyspark.sql import SparkSession, Row

# Inisialisasi findspark agar Python mengenali PySpark lokal
findspark.init()

# --- Inisiasi Sesi Spark Lokal ---
spark = SparkSession.builder \
    .appName("ABD_Tubes_Lokal") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Sesi Spark berhasil dibuat. Versi:", spark.version)

# --- Konfigurasi Path Utama ---
base_dir = "D:/ABD_Tubes"
raw_data_dir = os.path.join(base_dir, "data_DeepGlobe")
img_dir = os.path.join(raw_data_dir, "images")
mask_dir = os.path.join(raw_data_dir, "masks")

bronze_path = os.path.join(base_dir, "bronze_layer", "bronze.parquet")

sat_files = [f for f in os.listdir(img_dir) if f.endswith('_sat.jpg')]
target_base = sat_files[0].replace('_sat.jpg', '')

print(f"Membaca gambar mentah: {target_base}...")
img = cv2.imread(os.path.join(img_dir, f"{target_base}_sat.jpg"))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
mask = cv2.imread(os.path.join(mask_dir, f"{target_base}_mask.png"))
mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

# ==========================================
# 1. BRONZE LAYER (Ingestion Data Mentah)
# ==========================================
print("Memproses Bronze Layer...")
img_flat = img.reshape(-1, 3)
mask_flat = mask.reshape(-1, 3)

# Menggabungkan RGB citra dan RGB masker sebagai data mentah (Raw)
rows_bronze = [Row(R_sat=int(img_flat[i][0]), G_sat=int(img_flat[i][1]), B_sat=int(img_flat[i][2]), 
                   R_mask=int(mask_flat[i][0]), G_mask=int(mask_flat[i][1]), B_mask=int(mask_flat[i][2])) 
               for i in range(len(img_flat))]

df_bronze = spark.createDataFrame(rows_bronze)
df_bronze.write.mode("overwrite").parquet(bronze_path)
print(f"-> Berhasil menyimpan Bronze Layer ke: {bronze_path}")

# Hapus variabel yang sudah tidak dipakai untuk menghemat RAM
del rows_bronze, df_bronze

In [ ]:
import os
import numpy as np
from pyspark.sql import Row

# --- Konfigurasi Path Silver ---
silver_path = "D:/ABD_Tubes/silver_layer/silver.parquet"

# ==========================================
# 2. SILVER LAYER (Cleansing & Transformation)
# ==========================================
print("Memproses Silver Layer...")

# Membaca kembali data dari Bronze Layer secara terdistribusi
df_bronze_read = spark.read.parquet(bronze_path)

# Kamus warna untuk segmentasi kelas objek pada gambar
color_dict = {
    0: [0, 255, 255],  # Urban (Kuning/Cyan)
    1: [255, 255, 0],  # Agriculture (Kuning)
    2: [255, 0, 255],  # Rangeland (Magenta)
    3: [0, 0, 255],    # Water (Biru)
    4: [255, 255, 255] # Unknown/Barren (Putih)
}

# Transformasi logika: Menerjemahkan kombinasi RGB masker menjadi label kelas tunggal
label_flat = np.full(mask_flat.shape[0], 4)
for label, color in color_dict.items():
    matches = np.all(mask_flat == color, axis=1)
    label_flat[matches] = label

# Membuat skema Silver (Fitur RGB citra satelit yang bersih + Label Kelas)
rows_silver = [Row(R=int(img_flat[i][0]), G=int(img_flat[i][1]), B=int(img_flat[i][2]), Label=int(label_flat[i])) for i in range(len(img_flat))]
df_silver = spark.createDataFrame(rows_silver)

# Menulis data Silver terstruktur ke penyimpanan lokal berformat Parquet
df_silver.write.mode("overwrite").parquet(silver_path)
print(f"-> Berhasil menyimpan Silver Layer ke: {silver_path}")

# Manajemen Memori: Menghapus variabel array besar dari memori utama (RAM)
del img_flat, mask_flat, label_flat, rows_silver, df_bronze_read, df_silver

In [ ]:
from pyspark.ml.feature import VectorAssembler

# --- Konfigurasi Path Gold ---
gold_path = "D:/ABD_Tubes/gold_layer/gold.parquet"

# ==========================================
# 3. GOLD LAYER (Feature Engineering)
# ==========================================
print("Memproses Gold Layer...")

# Membaca data yang sudah dibersihkan dari Silver Layer
df_silver_read = spark.read.parquet(silver_path)

# Merakit kolom R, G, dan B menjadi satu kolom vektor tunggal
assembler = VectorAssembler(inputCols=["R", "G", "B"], outputCol="features")
df_gold = assembler.transform(df_silver_read)

# Menulis data Gold yang siap latih ke penyimpanan lokal
df_gold.write.mode("overwrite").parquet(gold_path)
print(f"-> Berhasil menyimpan Gold Layer ke: {gold_path}")

# Menghapus variabel sementara
del df_silver_read

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.mllib.evaluation import MulticlassMetrics

# ==========================================
# 4. MODEL TRAINING & EVALUATION (Random Forest)
# ==========================================
print("Memulai proses pemodelan Machine Learning...")

# Membaca data siap latih dari Gold Layer (sudah terdistribusi)
df_ready_to_train = spark.read.parquet(gold_path)

# Membagi dataset menjadi Data Latih (80%) dan Data Uji (20%)
train_data, test_data = df_ready_to_train.randomSplit([0.8, 0.2], seed=42)

print("Melatih model Random Forest (ini mungkin memakan waktu beberapa menit)...")
# Inisialisasi algoritma Random Forest dengan 20 pohon keputusan
rf = RandomForestClassifier(labelCol="Label", featuresCol="features", numTrees=20, maxDepth=10, seed=42)

# Eksekusi pelatihan model menggunakan data latih
rf_model = rf.fit(train_data)
print("-> Pelatihan model selesai!")

# Evaluasi Model menggunakan Data Uji
print("Menguji model dan menghitung metrik akurasi...")
predictions_test = rf_model.transform(test_data)

# Konversi DataFrame hasil prediksi ke format RDD yang dibutuhkan oleh MulticlassMetrics
pred_labels_rdd = predictions_test.selectExpr("cast(prediction as float)", "cast(Label as float)").rdd.map(tuple)
metrics = MulticlassMetrics(pred_labels_rdd)

print(f"\n======================================")
print(f"=> AKURASI PENGUJIAN: {metrics.accuracy * 100:.2f}%")
print(f"======================================\n")